# Enterprise Network Security System - Model Training Pipeline

This Jupyter Notebook outlines the machine learning pipeline used by the **Enterprise Network Security System** to train its classifiers (Random Forest, SVM) and anomaly detector (DBSCAN) on network traffic datasets.

## 1. Imports and Configurations

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set seeds for reproducibility
np.random.seed(42)

print("All libraries imported successfully!")

## 2. Load the Raw Cybersecurity Dataset

We will load the raw labeled dataset `uploads/network_dataset_labeled.csv` to inspect the network traffic features.

In [ ]:
dataset_path = 'uploads/network_dataset_labeled.csv'
if not os.path.exists(dataset_path):
    # Fallback to project root uploads if run from a different CWD
    dataset_path = '../project/backend/uploads/network_dataset_labeled.csv'

df = pd.read_csv(dataset_path)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 3. Preprocessing and Feature Selection

In [ ]:
# Dynamically identify the label/target column
def find_label_column(data):
    target_names = ['label', 'class', 'anomaly', 'target', 'attack', 'attack_type', 'type', 'status', 'classification']
    for col in data.columns:
        if col.lower() in target_names and col.lower() not in ['network target', 'video target']:
            return col
    return data.columns[-1]

label_col = find_label_column(df)
print(f"Detected label/target column: {label_col}")

# Identify feature columns
def get_feature_columns(data, target_col):
    feature_cols = []
    for col in data.columns:
        if col == target_col:
            continue
        col_lower = col.lower()
        exclude_keywords = ['id', 'uuid', 'timestamp', 'datetime', 'date', 'time', 'src_ip', 'dst_ip', 'ip_addr', 'ip']
        if any(kw in col_lower for kw in exclude_keywords):
            continue
        feature_cols.append(col)
    return feature_cols

feature_cols = get_feature_columns(df, label_col)
print(f"Features used for training ({len(feature_cols)}): {feature_cols}")

## 4. Custom Cybersecurity Classification Rules & DBSCAN

We define the rules to categorize network packets into: 
- **DDoS (1)**
- **Network Flooding (2)**
- **Packet Drop Attack (3)**
- **Jitter Attack (4)**
- **Performance Anomaly (5)** (using a pre-fitted DBSCAN clustering outlier checks)
- **Normal (0)**

In [ ]:
def apply_rules_and_dbscan(data):
    # Pre-scale features to run DBSCAN outlier clustering
    X_data = []
    for col in feature_cols:
        series = data[col]
        if not pd.api.types.is_numeric_dtype(series):
            series_str = series.fillna('missing').astype(str)
            unique_vals = sorted(list(series_str.unique()))
            mapping = {val: idx for idx, val in enumerate(unique_vals)}
            X_data.append(series_str.map(mapping).values)
        else:
            X_data.append(series.fillna(0.0).astype(float).values)
            
    X = np.column_stack(X_data)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    
    # DBSCAN eps=0.8, min_samples=10
    db = DBSCAN(eps=0.8, min_samples=10).fit(Xs)
    dbscan_labels = db.labels_
    
    y_labels = []
    for idx, row in data.iterrows():
        packet_loss = float(row.get('packet_loss', 0))
        latency = float(row.get('latency', 0))
        congestion = float(row.get('congestion', 0))
        throughput = float(row.get('throughput', 0))
        jitter = float(row.get('jitter', 0))
        is_dbscan_abnormal = (dbscan_labels[idx] == -1)
        
        # Custom rule conditions:
        if packet_loss > 30.0 or latency > 500.0 or congestion > 100.0:
            y_labels.append(1)  # DoS / DDoS
        elif congestion > 70.0 and throughput > 5.0:
            y_labels.append(2)  # Network Flooding
        elif latency > 100.0 or is_dbscan_abnormal:
            y_labels.append(5)  # Performance Anomaly / DBSCAN outlier
        elif 15.0 <= packet_loss <= 30.0:
            y_labels.append(3)  # Packet Drop Attack
        elif jitter > 5.0:
            y_labels.append(4)  # Jitter Attack
        else:
            y_labels.append(0)  # Normal
            
    return X, np.array(y_labels), scaler

X, y, scaler = apply_rules_and_dbscan(df)
unique, counts = np.unique(y, return_counts=True)
print("Class distribution after rule-based labeling:", dict(zip(unique, counts)))

## 5. Model Training & Evaluation (Random Forest vs SVM)

We split the labeled features into train and test sets to evaluate performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# Scale the features
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Random Forest Classifier
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

print("=== Random Forest Classifier Accuracy ===")
print(f"{accuracy_score(y_test, rf_pred) * 100:.2f}%")

# 2. Support Vector Machine (SVM) Classifier
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)

print("\n=== Support Vector Machine (SVM) Accuracy ===")
print(f"{accuracy_score(y_test, svm_pred) * 100:.2f}%")

## 6. Confusion Matrix Visualization

In [ ]:
class_names = ['Normal', 'DoS/DDoS', 'Flooding', 'Packet Drop', 'Jitter', 'Anomaly']

# Get actual classes present in test data
present_classes = np.unique(np.concatenate([y_test, rf_pred]))
labels_to_show = [class_names[c] for c in present_classes]

cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_to_show, yticklabels=labels_to_show)
plt.title('Random Forest - Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

## 7. Model Serialization

Here we save the trained models, preprocessing scaling weights, and metadata keys into pickle files.

In [ ]:
models_dir = 'models/'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(rf, os.path.join(models_dir, 'random_forest.pkl'))
joblib.dump(svm, os.path.join(models_dir, 'svm.pkl'))
joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))

print("Trained model files saved successfully to models/ directory!")